## 1. List geoJSON files
    - input: list of states
    - if _a and _b, then skip and add to "split" list

In [1]:
import os
import pandas as pd
from pathlib import Path
import re
from datetime import datetime

In [14]:
BASEDIR = Path('/mnt/beegfs/hellgate/home/vc149353/osm_fitness/')
STATE_FOLDERS = [f for f in BASEDIR.iterdir() if "700k" in f.name]  # l = 49

assert len(STATE_FOLDERS) == 49, f"Expected 49 state folders, found {len(STATE_FOLDERS)}"

In [ ]:
def list_submitted_and_split(state_folder):

    geojson_list = [f for f in state_folder.iterdir()]
    stem_list = [f.stem for f in geojson_list]

    submitted = []
    split = []

    for place in geojson_list:
        if place.stem[-2:] in ['_a', '_b']:
            submitted.append((state, place.stem))
        else:
            if f"{place.stem}_a" not in stem_list:  # wlg - a and b should always come togeher
                submitted.append((state, place.stem))
            else:
                split.append((state, place.stem))   

    assert len(geojson_list) == (len(submitted) + len(split))

    return submitted, split

In [67]:
state_list = ["Colorado", "Oklahoma", "Utah", "California", "Nevada", "New_Mexico", "North_Carolina", "Arizona"]
submitted = []
split = []

for s in state_list:
    s, spl = list_submitted_and_split(s)
    submitted += s
    split += spl


In [68]:
print(len(submitted))
print(len(split))

448
94


## 2b. Rename _North_* and _New_*
- North Carolina, New Mexico

In [97]:
def to_change(file_path):
    changes = [('North_expanded_cel_cdl_report.zip', 'North_Carolina_expanded_cel_cdl_report.zip'),
                ('North_pin_report.tsv.gz', 'North_Carolina_pin_report.tsv.gz'),
                ('New_expanded_cel_cdl_report.zip', 'New_Mexico_expanded_cel_cdl_report.zip'),
                ('New_pin_report.tsv.gz', 'New_Mexico_pin_report.tsv.gz'),
                ('North_pin_report.zip', 'North_Carolina_pin_report.zip'),
                ('New_pin_report.zip', 'New_Mexico_pin_report.zip'),
            ]

    for old, new in changes:
        if file_path.name.endswith(old):
            return (old, new)
    
    return False

In [172]:
downloads_path = Path('/mnt/beegfs/hellgate/home/vc149353/azira_downloads')
downloads_list = downloads_path.iterdir()

for f in downloads_list:
    if (name_update := to_change(f)):
        old, new = name_update
        print(f.name)
        start_idx = f.name.find(old)
        new_name = f"{f.name[:start_idx]}{new}"
        f.rename(f.with_name(new_name))

## 2. For submitted files, check if downloaded

In [ ]:
# intended to replace ' ', '.', '(', and ')' with '_'
# problems are El Pasa de Robles (Paso Robles), St. George, San Buenaventrua (Ventura)
def clean_stem(stem):
    return re.sub(r'[^A-Za-z0-9_-]', '_', stem)

In [181]:
downloads_path = Path('/mnt/beegfs/hellgate/home/vc149353/azira_downloads')
download_list = [f for f in downloads_path.iterdir()]

downloaded = []
not_downloaded = []

for state, name in submitted:
    pin_flag = False
    cel_cdl_flag = False
    clean_name = clean_stem(name)
    for d in download_list:
        if state in d.name and clean_name in d.name:
            downloaded.append({
                "job_id": d.name.split("_")[0],
                "state": state, 
                "name": clean_name, 
                "filename": d.name, 
                "size_mb": d.stat().st_size / (1024 ** 2),
                "created":  datetime.fromtimestamp(d.stat().st_ctime)
            })
            
            if d.name.endswith("pin_report.tsv.gz") or d.name.endswith('pin_report.zip'):
                pin_flag = True
            elif d.name.endswith("expanded_cel_cdl_report.zip"):
                cel_cdl_flag = True

            if pin_flag and cel_cdl_flag:
                break
    if not (pin_flag and cel_cdl_flag):
        not_downloaded.append((state, clean_name, d.name))

print(f"Submitted: {len(submitted)}")
print(f"Downloaded: {len(downloaded)}")
print(f"Not Downloaded: {len(not_downloaded)}")

df_downloaded = pd.DataFrame(downloaded)
df_downloaded.sort_values(by=['state', 'name'], inplace=True)
df_downloaded.reset_index(drop=True, inplace=True)
df_downloaded.to_csv("reports_summary_AZ_CA_CO_NV_NM_NC_OK_UT.csv", index=False)

Submitted: 448
Downloaded: 897
Not Downloaded: 0


In [183]:
df_downloaded['size_mb'].sum()

np.float64(582480.1366300583)

In [186]:
df_downloaded.groupby(by=['job_id']).count()

,state,name,filename,size_mb,created
job_id,,,,,
.10056850,1,1,1,1,1
10056849,2,2,2,2,2
10056850,2,2,2,2,2
10056851,2,2,2,2,2
10056852,2,2,2,2,2
...,...,...,...,...,...
10057735,2,2,2,2,2
10057736,2,2,2,2,2
10057737,2,2,2,2,2
